# Dual Dataset GraphRAG Experiment

Notebook запускает один воспроизводимый эксперимент для двух датасетов:

- MultiHop-RAG
- MuSiQue-Ans

Для каждого датасета выполняются загрузка, индексация, full LLM enrichment и retrieval-side ablation. Шаги пропускаются, если соответствующие артефакты уже существуют.


## 1. Конфигурация окружения


In [1]:
import os
import sys
import json
from pathlib import Path

import pandas as pd
from IPython.display import display, FileLink

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / 'src'))

# .env перезагружается явно. После изменения .env повторно выполните эту ячейку.
env_path = PROJECT_ROOT / '.env'
if env_path.exists():
    for raw in env_path.read_text(encoding='utf-8').splitlines():
        line = raw.strip()
        if line and not line.startswith('#') and '=' in line:
            key, value = line.split('=', 1)
            os.environ[key.strip()] = value.strip().strip('"').strip("'")

from ecgraphrag.openrouter import OpenRouterConfig
openrouter_config = OpenRouterConfig.from_env()

LIMIT_DOCS = 50
LIMIT_QA = 25
MAX_LLM_UNITS = None
TOP_K = 10
CHUNK_SIZE = 600
OVERLAP = 100
RETRIEVAL_MODE = 'two_stage'

MAX_ENRICH_EDGES = None
MAX_ENRICH_ENTITIES = None
ENRICH_STEPS = ['questions', 'summaries', 'contradictions', 'entities', 'infer', 'importance']

SKIP_IF_DATASET_EXISTS = True
SKIP_IF_INDEX_EXISTS = True
SKIP_IF_ENRICHED_EXISTS = True
RUN_UNIT_TESTS = True

# False: считать ablation даже при failed chunks в gold evidence, но записывать предупреждение.
STRICT_FAILED_EVIDENCE = False

DATASETS = {
    'multihop_rag': {
        'data_dir': Path('data/multihop_rag'),
        'index_dir': Path('work/multihop_graphrag_openrouter'),
        'enriched_dir': Path('work/multihop_graphrag_enriched'),
        'limit_docs': LIMIT_DOCS,
        'limit_qa': LIMIT_QA,
        'max_llm_units': MAX_LLM_UNITS,
    },
    'musique_ans': {
        'data_dir': Path('data/musique_ans'),
        'index_dir': Path('work/musique_ans_graphrag_openrouter'),
        'enriched_dir': Path('work/musique_ans_graphrag_enriched'),
        'limit_docs': None,
        'limit_qa': LIMIT_QA,
        'max_llm_units': MAX_LLM_UNITS,
    },
}

ABLATION_OUTPUT_ROOT = Path('work/ablations')

print(f'Project root: {PROJECT_ROOT}')
print(f'OPENROUTER_API_KEY set: {bool(openrouter_config.api_key)}')
print(f'OpenRouter model: {openrouter_config.model}')
print(f'OpenRouter max_tokens: {openrouter_config.max_tokens}')
print(f'OpenRouter timeout: {openrouter_config.timeout}')
print(f'OpenRouter retries: {openrouter_config.retries}')
print(f'OpenRouter workers: {openrouter_config.workers}')
DATASETS


Project root: /home/kaa21/GraphRAGnew
OPENROUTER_API_KEY set: True
OpenRouter model: qwen/qwen3.6-flash
OpenRouter max_tokens: 10000
OpenRouter timeout: 90
OpenRouter retries: 3
OpenRouter workers: 12


{'multihop_rag': {'data_dir': PosixPath('data/multihop_rag'),
  'index_dir': PosixPath('work/multihop_graphrag_openrouter'),
  'enriched_dir': PosixPath('work/multihop_graphrag_enriched'),
  'limit_docs': 50,
  'limit_qa': 25,
  'max_llm_units': None},
 'musique_ans': {'data_dir': PosixPath('data/musique_ans'),
  'index_dir': PosixPath('work/musique_ans_graphrag_openrouter'),
  'enriched_dir': PosixPath('work/musique_ans_graphrag_enriched'),
  'limit_docs': None,
  'limit_qa': 25,
  'max_llm_units': None}}

## 2. Unit tests

Перед запуском эксперимента проверяется код проекта. Ячейку можно отключить через `RUN_UNIT_TESTS = False` в конфигурации.


In [2]:
if RUN_UNIT_TESTS:
    import subprocess

    env = os.environ.copy()
    env['PYTHONPATH'] = str(PROJECT_ROOT / 'src')
    result = subprocess.run(
        [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests', '-v'],
        cwd=PROJECT_ROOT,
        env=env,
        text=True,
        capture_output=True,
    )
    print(result.stdout)
    print(result.stderr)
    if result.returncode != 0:
        raise RuntimeError('Unit tests failed')
    print('Unit tests passed.')
else:
    print('Unit tests skipped.')



test_ablation_can_warn_instead_of_invalidating_failed_evidence (test_ablation.AblationTest.test_ablation_can_warn_instead_of_invalidating_failed_evidence) ... ok
test_ablation_outputs_all_variants_without_enrichment_call (test_ablation.AblationTest.test_ablation_outputs_all_variants_without_enrichment_call) ... ok
test_download_resolves_github_lfs_pointer (test_dataset.DatasetDownloadTest.test_download_resolves_github_lfs_pointer) ... ok
test_musique_normalization_keeps_supporting_evidence_covered (test_dataset.DatasetDownloadTest.test_musique_normalization_keeps_supporting_evidence_covered) ... ok
test_normalization_selects_qa_evidence_documents_first (test_dataset.DatasetDownloadTest.test_normalization_selects_qa_evidence_documents_first) ... ok
test_qa_splits_are_reproducible_and_stratified (test_dataset.DatasetDownloadTest.test_qa_splits_are_reproducible_and_stratified) ... ok
test_contradiction_detected (test_enrich.TestContradictionAnalysis.test_contradiction_detected)
LLM ident

## 3. Загрузка и нормализация датасетов

Если `documents.jsonl` и `qa.jsonl` уже существуют, шаг пропускается.


In [3]:
from ecgraphrag.dataset import download_multihop_rag_dataset, download_musique_ans_dataset


def dataset_ready(data_dir: Path) -> bool:
    return (data_dir / 'documents.jsonl').exists() and (data_dir / 'qa.jsonl').exists()


def load_manifest(data_dir: Path) -> dict:
    manifest_path = data_dir / 'dataset_manifest.json'
    return json.loads(manifest_path.read_text(encoding='utf-8')) if manifest_path.exists() else {}


dataset_manifests = {}
for dataset_name, cfg in DATASETS.items():
    data_dir = cfg['data_dir']
    if SKIP_IF_DATASET_EXISTS and dataset_ready(data_dir):
        manifest = load_manifest(data_dir)
        manifest['status'] = 'skipped'
        print(f'{dataset_name}: dataset exists, skipped: {data_dir}')
    elif dataset_name == 'multihop_rag':
        manifest = download_multihop_rag_dataset(
            data_dir,
            limit_docs=cfg['limit_docs'],
            limit_qa=cfg['limit_qa'],
        )
        manifest['status'] = 'created'
    elif dataset_name == 'musique_ans':
        manifest = download_musique_ans_dataset(
            data_dir,
            limit_qa=cfg['limit_qa'],
        )
        manifest['status'] = 'created'
    else:
        raise ValueError(dataset_name)
    dataset_manifests[dataset_name] = manifest

manifest_rows = []
for dataset_name, manifest in dataset_manifests.items():
    coverage = manifest.get('evidence_coverage', {})
    manifest_rows.append({
        'dataset': dataset_name,
        'status': manifest.get('status'),
        'documents': manifest.get('documents'),
        'qa': manifest.get('qa'),
        'complete_qa': coverage.get('complete_qa'),
        'partial_qa': coverage.get('partial_qa'),
        'total_qa': coverage.get('total_qa'),
    })

dataset_manifest_df = pd.DataFrame(manifest_rows)
display(dataset_manifest_df)


multihop_rag: dataset exists, skipped: data/multihop_rag
musique_ans: dataset exists, skipped: data/musique_ans


,dataset,status,documents,qa,complete_qa,partial_qa,total_qa
0,multihop_rag,skipped,50,25,20,20,25
1,musique_ans,skipped,444,25,25,25,25


## 4. Индексация baseline GraphRAG

Индексация использует LLM extractor и resume cache. Если `calibrated_edges.jsonl` уже существует, индекс переиспользуется.


In [4]:
from ecgraphrag.indexer import GraphRAGIndexer
from ecgraphrag.storage import read_jsonl


def index_ready(index_dir: Path) -> bool:
    return (index_dir / 'calibrated_edges.jsonl').exists()


index_counts = {}
for dataset_name, cfg in DATASETS.items():
    data_dir = cfg['data_dir']
    index_dir = cfg['index_dir']
    if SKIP_IF_INDEX_EXISTS and index_ready(index_dir):
        counts = {
            'status': 'skipped',
            'documents': len(read_jsonl(index_dir / 'documents.jsonl')),
            'text_units': len(read_jsonl(index_dir / 'text_units.jsonl')),
            'entities': len(read_jsonl(index_dir / 'entities.jsonl')),
            'calibrated_edges': len(read_jsonl(index_dir / 'calibrated_edges.jsonl')),
        }
        print(f'{dataset_name}: index exists, skipped: {index_dir}')
    else:
        indexer = GraphRAGIndexer(
            chunk_size=CHUNK_SIZE,
            overlap=OVERLAP,
            extractor='llm',
            max_llm_units=cfg['max_llm_units'],
            resume=True,
        )
        counts = indexer.index(data_dir / 'documents.jsonl', index_dir)
        counts['status'] = 'created'
    index_counts[dataset_name] = counts

index_df = pd.DataFrame(index_counts).T
display(index_df)


multihop_rag: index exists, skipped: work/multihop_graphrag_openrouter
musique_ans: index exists, skipped: work/musique_ans_graphrag_openrouter


,status,documents,text_units,entities,calibrated_edges
multihop_rag,skipped,50,223,2108,2440
musique_ans,skipped,444,444,3078,2955


## 5. Full LLM enrichment

Создаётся один full enriched index для каждого датасета. Если enriched index уже существует, LLM enrichment не запускается повторно.


In [5]:
from ecgraphrag.enrich import enrich_graph, finalize_enriched_index
from ecgraphrag.models import Edge, Entity
from ecgraphrag.openrouter import OpenRouterClient


def enriched_ready(enriched_dir: Path) -> bool:
    return (enriched_dir / 'calibrated_edges.jsonl').exists() and (enriched_dir / 'entities.jsonl').exists()


def load_graph(index_dir: Path) -> tuple[list[Entity], list[Edge]]:
    entities = [Entity(**row) for row in read_jsonl(index_dir / 'entities.jsonl')]
    edges = [Edge(**row) for row in read_jsonl(index_dir / 'calibrated_edges.jsonl')]
    return entities, edges


def enrich_dataset(dataset_name: str, cfg: dict) -> dict:
    index_dir = cfg['index_dir']
    enriched_dir = cfg['enriched_dir']
    if SKIP_IF_ENRICHED_EXISTS and enriched_ready(enriched_dir):
        edges = read_jsonl(enriched_dir / 'calibrated_edges.jsonl')
        entities = read_jsonl(enriched_dir / 'entities.jsonl')
        print(f'{dataset_name}: enriched index exists, skipped: {enriched_dir}')
        return {
            'status': 'skipped',
            'entities': len(entities),
            'calibrated_edges': len(edges),
            'inferred_edges': sum(row.get('evidence_type') == 'inferred' for row in edges),
        }

    all_entities, all_edges = load_graph(index_dir)
    edges_to_enrich = all_edges[:MAX_ENRICH_EDGES] if MAX_ENRICH_EDGES else all_edges
    entities_to_enrich = all_entities[:MAX_ENRICH_ENTITIES] if MAX_ENRICH_ENTITIES else all_entities

    print(f'{dataset_name}: enriching {len(edges_to_enrich)} edges and {len(entities_to_enrich)} entities')
    client = OpenRouterClient()
    enriched_entities_subset, enriched_edges_subset = enrich_graph(
        entities_to_enrich,
        edges_to_enrich,
        client,
        steps=ENRICH_STEPS,
    )

    enriched_edge_ids = {edge.id for edge in enriched_edges_subset}
    remaining_edges = [edge for edge in all_edges if edge.id not in enriched_edge_ids]
    enriched_edges = enriched_edges_subset + remaining_edges

    enriched_entity_ids = {entity.id for entity in enriched_entities_subset}
    remaining_entities = [entity for entity in all_entities if entity.id not in enriched_entity_ids]
    enriched_entities = enriched_entities_subset + remaining_entities

    counts = finalize_enriched_index(index_dir, enriched_dir, enriched_entities, enriched_edges)
    counts['status'] = 'created'
    counts['inferred_edges'] = sum(edge.evidence_type == 'inferred' for edge in enriched_edges)
    return counts


enriched_counts = {}
for dataset_name, cfg in DATASETS.items():
    enriched_counts[dataset_name] = enrich_dataset(dataset_name, cfg)

enriched_df = pd.DataFrame(enriched_counts).T
display(enriched_df)


multihop_rag: enriched index exists, skipped: work/multihop_graphrag_enriched


musique_ans: enriched index exists, skipped: work/musique_ans_graphrag_enriched


,status,entities,calibrated_edges,inferred_edges
multihop_rag,skipped,2053,2413,20
musique_ans,skipped,3030,2931,15


## 6. Ablation metrics

Ablation запускается без повторного LLM enrichment. Для каждого датасета используются baseline index и full enriched index, а retrieval включает разные комбинации enriched-сигналов.


In [6]:
import importlib
import ecgraphrag.benchmark as benchmark_module

importlib.invalidate_caches()
importlib.reload(benchmark_module)

run_retrieval_ablation = benchmark_module.run_retrieval_ablation
load_retrieval_config = benchmark_module.load_retrieval_config

ablation_results = {}
for dataset_name, cfg in DATASETS.items():
    output_dir = ABLATION_OUTPUT_ROOT / dataset_name
    result = run_retrieval_ablation(
        dataset_name=dataset_name,
        data_path=cfg['data_dir'],
        index_path=cfg['index_dir'],
        enriched_index_path=cfg['enriched_dir'],
        output_path=output_dir,
        config=load_retrieval_config(None),
        top_k=TOP_K,
        limit=cfg['limit_qa'],
        strict_failed_evidence=STRICT_FAILED_EVIDENCE,
    )
    ablation_results[dataset_name] = result
    print(f'{dataset_name}: ablation saved to {output_dir}')

print('Ablation complete.')


multihop_rag: ablation saved to work/ablations/multihop_rag


musique_ans: ablation saved to work/ablations/musique_ans
Ablation complete.


## 7. Сводная таблица ablation

`delta_*` считается относительно `calibrated_base_index` внутри того же датасета.


In [7]:
metrics = [
    'all_evidence_success_at_k',
    'recall_at_k',
    'precision_at_k',
    'mrr',
    'ndcg_at_k',
    'packed_context_recall_at_k',
    'answer_hit_rate',
    'latency_ms_p50',
    'latency_ms_p95',
]

rows = []
for dataset_name, result in ablation_results.items():
    for variant, summary in result['summary'].items():
        row = {
            'dataset': dataset_name,
            'variant': variant,
            'retrieval_mode': summary.get('retrieval_mode'),
            'valid': summary.get('valid', True),
            'signals': '+'.join(summary.get('signals', [])),
            'count': summary.get('count'),
            'total_qa': summary.get('total_qa'),
        }
        for metric in metrics:
            row[metric] = summary.get(metric, 0.0)
            row[f'delta_{metric}'] = summary.get('delta_vs_calibrated', {}).get(metric, 0.0)
        rows.append(row)

ablation_summary = pd.DataFrame(rows)
display(ablation_summary)

for dataset_name in DATASETS:
    output_dir = ABLATION_OUTPUT_ROOT / dataset_name
    display(FileLink(str((output_dir / 'summary.csv').resolve())))
    display(FileLink(str((output_dir / 'summary.json').resolve())))


,dataset,variant,retrieval_mode,valid,signals,count,total_qa,all_evidence_success_at_k,delta_all_evidence_success_at_k,recall_at_k,...,ndcg_at_k,delta_ndcg_at_k,packed_context_recall_at_k,delta_packed_context_recall_at_k,answer_hit_rate,delta_answer_hit_rate,latency_ms_p50,delta_latency_ms_p50,latency_ms_p95,delta_latency_ms_p95
0,multihop_rag,baseline_base_index,two_stage,True,,20,25,0.80,0.00,0.925000,...,0.866732,0.000000,0.925000,0.000000,1.00,0.00,335.230,4.984,485.762,22.837
1,multihop_rag,calibrated_base_index,two_stage,True,,20,25,0.80,0.00,0.925000,...,0.866732,0.000000,0.925000,0.000000,1.00,0.00,330.246,0.000,462.925,0.000
2,multihop_rag,enriched_control_no_fields,two_stage,True,,20,25,0.80,0.00,0.925000,...,0.866732,0.000000,0.925000,0.000000,1.00,0.00,333.245,2.999,464.313,1.388
3,multihop_rag,questions,two_stage,True,questions,20,25,0.90,0.10,0.954167,...,0.896386,0.029654,0.937500,0.012500,1.00,0.00,452.762,122.516,542.954,80.029
4,multihop_rag,summaries,two_stage,True,summaries,20,25,0.85,0.05,0.937500,...,0.889508,0.022776,0.925000,0.000000,1.00,0.00,355.267,25.021,570.823,107.898
5,multihop_rag,entities,two_stage,True,entities,20,25,0.85,0.05,0.941667,...,0.895001,0.028269,0.941667,0.016667,1.00,0.00,506.762,176.516,692.075,229.150
6,multihop_rag,inferred_edges,two_stage,True,inferred_edges,20,25,0.80,0.00,0.925000,...,0.837560,-0.029172,0.908333,-0.016667,1.00,0.00,419.902,89.656,560.214,97.289
7,multihop_rag,importance,two_stage,True,importance,20,25,0.80,0.00,0.925000,...,0.866732,0.000000,0.925000,0.000000,1.00,0.00,397.694,67.448,527.582,64.657
8,multihop_rag,contradiction_penalty,two_stage,True,contradiction_penalty,20,25,0.80,0.00,0.925000,...,0.866732,0.000000,0.925000,0.000000,1.00,0.00,396.050,65.804,497.956,35.031
9,multihop_rag,questions+summaries,two_stage,True,questions+summaries,20,25,0.90,0.10,0.954167,...,0.893597,0.026865,0.925000,0.000000,1.00,0.00,416.523,86.277,674.005,211.080


/home/kaa21/GraphRAGnew/work/ablations/multihop_rag/summary.csv

/home/kaa21/GraphRAGnew/work/ablations/multihop_rag/summary.json

/home/kaa21/GraphRAGnew/work/ablations/musique_ans/summary.csv

/home/kaa21/GraphRAGnew/work/ablations/musique_ans/summary.json

## 8. Enrichment statistics


In [8]:
stats_rows = []
for dataset_name, cfg in DATASETS.items():
    enriched_dir = cfg['enriched_dir']
    edges = read_jsonl(enriched_dir / 'calibrated_edges.jsonl')
    entities = read_jsonl(enriched_dir / 'entities.jsonl')
    stats_rows.append({
        'dataset': dataset_name,
        'edges': len(edges),
        'entities': len(entities),
        'with_generated_questions': sum(bool(row.get('generated_questions')) for row in edges),
        'with_semantic_summary': sum(bool(row.get('semantic_summary')) for row in edges),
        'with_contradiction_info': sum(bool(row.get('contradiction_info')) for row in edges),
        'inferred_edges': sum(row.get('evidence_type') == 'inferred' for row in edges),
        'importance_gt_0_7': sum(float(row.get('importance') or 0.0) > 0.7 for row in edges),
        'enriched_entities': sum(bool(row.get('enriched_description') or row.get('aliases') or row.get('category')) for row in entities),
    })

enrichment_stats = pd.DataFrame(stats_rows)
display(enrichment_stats)


,dataset,edges,entities,with_generated_questions,with_semantic_summary,with_contradiction_info,inferred_edges,importance_gt_0_7,enriched_entities
0,multihop_rag,2413,2053,2393,2393,4,20,1068,2038
1,musique_ans,2931,3030,2916,2916,0,15,1387,3002
